In [ ]:
import pandas as pd
import torch
import numpy as np
import time
import os
import json
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import gc
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

class Config:
    train_path = "/home/aman_swaraj/Downloads/Codelite/train_original.csv"
    test_path = "/home/aman_swaraj/Downloads/Codelite/test_original.csv"
    tag_vocab_path = "/home/aman_swaraj/Downloads/Codelite/unique_tags.csv"
    
    max_length = 512
    batch_size = 16
    eval_batch_size = 8
    num_epochs = 3
    learning_rate = 2e-5
    device = "cuda" if torch.cuda.is_available() else "cpu"
    seed = 42
    
    models_to_test = {
        "distilbert": "distilbert-base-uncased",
        "albert": "albert-base-v2",
        "mobilebert": "google/mobilebert-uncased",
        "tinybert": "huawei-noah/TinyBERT_General_4L_312D",
        "electra": "google/electra-small-discriminator",
        "deberta": "microsoft/deberta-v3-base",
    }
    
    results_dir = "/home/aman_swaraj/Downloads/Codelite/experiment_results"
    models_dir = "/home/aman_swaraj/Downloads/Codelite/saved_models"

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(Config.seed)

os.makedirs(Config.results_dir, exist_ok=True)
os.makedirs(Config.models_dir, exist_ok=True)

print("Loading datasets...")
train_data = pd.read_csv(Config.train_path)
test_data = pd.read_csv(Config.test_path)
tag_vocab = pd.read_csv(Config.tag_vocab_path)["Tag"].tolist()

label_encoder = LabelEncoder()
label_encoder.fit(tag_vocab)
train_data["EncodedTags"] = label_encoder.transform(train_data["language"])
test_data["EncodedTags"] = label_encoder.transform(test_data["language"])

num_classes = len(tag_vocab)
print(f"Number of classes: {num_classes}")
print(f"Train samples: {len(train_data)}, Test samples: {len(test_data)}")

class CodeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        code = str(row["code"])
        
        encoding = self.tokenizer(
            code,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(row["EncodedTags"], dtype=torch.long)
        }

class CodeClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout_rate=0.1):
        super(CodeClassifier, self).__init__()
        
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_size, num_classes)
        
        self.model_name = model_name
        
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        last_hidden_state = outputs.last_hidden_state
        
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask
        
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        
        return logits
    
    def get_parameter_count(self):
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total_params, trainable_params

def train_model(model_name, model_path, tokenizer_name=None):
    if tokenizer_name is None:
        tokenizer_name = model_name
    
    print(f"\n{'='*80}")
    print(f"Training: {model_name}")
    print(f"{'='*80}")
    
    start_time = time.time()
    
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    
    train_dataset = CodeDataset(train_data, tokenizer, Config.max_length)
    test_dataset = CodeDataset(test_data, tokenizer, Config.max_length)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=Config.eval_batch_size, shuffle=False)
    
    model = CodeClassifier(model_name, num_classes).to(Config.device)
    
    total_params, trainable_params = model.get_parameter_count()
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    optimizer = optim.AdamW(model.parameters(), lr=Config.learning_rate, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=len(train_loader) * Config.num_epochs)
    criterion = nn.CrossEntropyLoss()
    
    train_losses = []
    train_accuracies = []
    
    for epoch in range(Config.num_epochs):
        model.train()
        epoch_loss = 0
        correct = 0
        total = 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.num_epochs}")
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(Config.device)
            attention_mask = batch["attention_mask"].to(Config.device)
            labels = batch["labels"].to(Config.device)
            
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()
            
            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            epoch_loss += loss.item()
            
            progress_bar.set_postfix({
                'loss': loss.item(),
                'acc': f'{100 * correct / total:.2f}%'
            })
        
        avg_loss = epoch_loss / len(train_loader)
        accuracy = 100 * correct / total
        train_losses.append(avg_loss)
        train_accuracies.append(accuracy)
        
        print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}, Accuracy = {accuracy:.2f}%")
    
    training_time = time.time() - start_time
    
    model_save_path = os.path.join(Config.models_dir, f"{model_name.replace('/', '_')}_model.pth")
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': model.encoder.config,
        'label_encoder': label_encoder,
        'model_name': model_name
    }, model_save_path)
    print(f"Model saved to: {model_save_path}")
    
    print("Evaluating on test set...")
    model.eval()
    
    inference_start = time.time()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(Config.device)
            attention_mask = batch["attention_mask"].to(Config.device)
            labels = batch["labels"].cpu().numpy()
            
            logits = model(input_ids, attention_mask)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels)
            all_probs.extend(probs.cpu().numpy())
    
    inference_time = time.time() - inference_start
    avg_inference_time = inference_time / len(test_data)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
    
    class_report = classification_report(all_labels, all_preds, target_names=tag_vocab, output_dict=True)
    
    if torch.cuda.is_available():
        memory_allocated = torch.cuda.max_memory_allocated() / 1024**3
        memory_reserved = torch.cuda.max_memory_reserved() / 1024**3
    else:
        memory_allocated = memory_reserved = 0
    
    del model, tokenizer
    torch.cuda.empty_cache() if torch.cuda.is_available() else gc.collect()
    
    results = {
        'model_name': model_name,
        'total_parameters': total_params,
        'trainable_parameters': trainable_params,
        'training_time_seconds': training_time,
        'inference_time_total': inference_time,
        'avg_inference_time_per_sample': avg_inference_time,
        'accuracy': accuracy,
        'weighted_precision': precision,
        'weighted_recall': recall,
        'weighted_f1': f1,
        'peak_memory_gb': memory_allocated,
        'reserved_memory_gb': memory_reserved,
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'class_report': class_report,
        'predictions': all_preds,
        'labels': all_labels,
        'model_size_mb': os.path.getsize(model_save_path) / 1024**2 if os.path.exists(model_save_path) else 0
    }
    
    print(f"\nResults for {model_name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  Training Time: {training_time:.2f}s")
    print(f"  Avg Inference Time: {avg_inference_time:.4f}s per sample")
    print(f"  Model Size: {results['model_size_mb']:.2f} MB")
    
    return results

def run_all_experiments():
    all_results = {}
    
    special_cases = {
        "deberta": {"tokenizer": "microsoft/deberta-v3-base"},
    }
    
    for model_key, model_name in Config.models_to_test.items():
        try:
            print(f"\n{'#'*100}")
            print(f"Starting experiment {len(all_results) + 1}/{len(Config.models_to_test)}: {model_key}")
            print(f"{'#'*100}")
            
            if model_key in special_cases:
                tokenizer_name = special_cases[model_key]["tokenizer"]
            else:
                tokenizer_name = model_name
            
            results = train_model(model_name, model_key, tokenizer_name)
            all_results[model_key] = results
            
            with open(os.path.join(Config.results_dir, f"{model_key}_results.json"), 'w') as f:
                json.dump({k: v for k, v in results.items() 
                          if k not in ['predictions', 'labels', 'class_report', 'train_losses', 'train_accuracies']}, 
                         f, indent=2, default=str)
            
            class_report_df = pd.DataFrame(results['class_report']).transpose()
            class_report_df.to_csv(os.path.join(Config.results_dir, f"{model_key}_class_report.csv"))
            
        except Exception as e:
            print(f"Error training {model_key}: {str(e)}")
            all_results[model_key] = {"error": str(e)}
            continue
    
    return all_results

def create_comparison_plots(all_results):
    successful_models = {k: v for k, v in all_results.items() if 'error' not in v}
    
    if not successful_models:
        print("No successful models to plot")
        return
    
    plot_data = []
    for model_name, results in successful_models.items():
        plot_data.append({
            'Model': model_name,
            'Accuracy': results['accuracy'],
            'F1-Score': results['weighted_f1'],
            'Training Time (s)': results['training_time_seconds'],
            'Inference Time/sample (ms)': results['avg_inference_time_per_sample'] * 1000,
            'Model Size (MB)': results['model_size_mb'],
            'Parameters (M)': results['total_parameters'] / 1e6,
            'Trainable Parameters (M)': results['trainable_parameters'] / 1e6
        })
    
    df = pd.DataFrame(plot_data)
    
    plots_dir = os.path.join(Config.results_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)
    
    plt.style.use('seaborn-v0_8-darkgrid')
    sns.set_palette("husl")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    ax1.bar(df['Model'], df['Accuracy'])
    ax1.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.set_ylim([0, 1])
    ax1.tick_params(axis='x', rotation=45)
    
    for i, v in enumerate(df['Accuracy']):
        ax1.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')
    
    ax2.bar(df['Model'], df['F1-Score'])
    ax2.set_title('Model F1-Score Comparison', fontsize=14, fontweight='bold')
    ax2.set_ylabel('F1-Score', fontsize=12)
    ax2.set_ylim([0, 1])
    ax2.tick_params(axis='x', rotation=45)
    
    for i, v in enumerate(df['F1-Score']):
        ax2.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'accuracy_f1_comparison.png'), dpi=300, bbox_inches='tight')
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    
    ax1.bar(df['Model'], df['Training Time (s)'])
    ax1.set_title('Training Time Comparison', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Time (seconds)', fontsize=12)
    ax1.tick_params(axis='x', rotation=45)
    
    ax2.bar(df['Model'], df['Inference Time/sample (ms)'])
    ax2.set_title('Inference Time per Sample', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Time (milliseconds)', fontsize=12)
    ax2.tick_params(axis='x', rotation=45)
    
    ax3.bar(df['Model'], df['Model Size (MB)'])
    ax3.set_title('Model Size Comparison', fontsize=14, fontweight='bold')
    ax3.set_ylabel('Size (MB)', fontsize=12)
    ax3.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'efficiency_metrics.png'), dpi=300, bbox_inches='tight')
    
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(df))
    width = 0.35
    
    ax.bar(x - width/2, df['Parameters (M)'], width, label='Total Parameters')
    ax.bar(x + width/2, df['Trainable Parameters (M)'], width, label='Trainable Parameters')
    
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('Parameters (Millions)', fontsize=12)
    ax.set_title('Parameter Count Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(df['Model'], rotation=45)
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'parameter_comparison.png'), dpi=300, bbox_inches='tight')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    scatter = ax.scatter(df['Model Size (MB)'], df['Accuracy'], 
                         s=df['Parameters (M)']*10, alpha=0.6, 
                         c=df['Inference Time/sample (ms)'], cmap='viridis')
    
    for i, row in df.iterrows():
        ax.annotate(row['Model'], (row['Model Size (MB)'], row['Accuracy']), 
                   fontsize=8, alpha=0.7)
    
    ax.set_xlabel('Model Size (MB)', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title('Accuracy vs Model Size (bubble size = parameters, color = inference time)', 
                 fontsize=14, fontweight='bold')
    
    cbar = plt.colorbar(scatter)
    cbar.set_label('Inference Time per Sample (ms)', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'accuracy_vs_efficiency.png'), dpi=300, bbox_inches='tight')
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    for model_name, results in successful_models.items():
        if 'train_losses' in results and len(results['train_losses']) > 0:
            ax1.plot(range(1, len(results['train_losses']) + 1), 
                    results['train_losses'], 
                    marker='o', linewidth=2, label=model_name)
    
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training Loss Curves', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    for model_name, results in successful_models.items():
        if 'train_accuracies' in results and len(results['train_accuracies']) > 0:
            ax2.plot(range(1, len(results['train_accuracies']) + 1), 
                    results['train_accuracies'], 
                    marker='s', linewidth=2, label=model_name)
    
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training Accuracy Curves', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, 'training_curves.png'), dpi=300, bbox_inches='tight')
    
    plt.close('all')
    print(f"Plots saved to: {plots_dir}")

def generate_summary_report(all_results):
    successful_models = {k: v for k, v in all_results.items() if 'error' not in v}
    
    if not successful_models:
        print("No successful models to generate report")
        return
    
    summary_data = []
    for model_name, results in successful_models.items():
        summary_data.append({
            'Model': model_name,
            'Accuracy': results['accuracy'],
            'Precision': results['weighted_precision'],
            'Recall': results['weighted_recall'],
            'F1-Score': results['weighted_f1'],
            'Total Parameters': results['total_parameters'],
            'Trainable Parameters': results['trainable_parameters'],
            'Training Time (s)': results['training_time_seconds'],
            'Total Inference Time (s)': results['inference_time_total'],
            'Avg Inference Time/sample (ms)': results['avg_inference_time_per_sample'] * 1000,
            'Model Size (MB)': results['model_size_mb'],
            'Peak Memory (GB)': results['peak_memory_gb']
        })
    
    df_summary = pd.DataFrame(summary_data)
    df_summary = df_summary.sort_values('Accuracy', ascending=False)
    
    summary_path = os.path.join(Config.results_dir, "model_comparison_summary.csv")
    df_summary.to_csv(summary_path, index=False)
    print(f"Summary saved to: {summary_path}")
    
    print("\n" + "="*120)
    print("MODEL COMPARISON SUMMARY")
    print("="*120)
    print(df_summary.to_string(index=False))
    print("="*120)
    
    print("\n" + "="*120)
    print("RECOMMENDATIONS")
    print("="*120)
    
    best_acc = df_summary.iloc[0]
    print(f"📈 Best Accuracy: {best_acc['Model']} with {best_acc['Accuracy']:.4f} accuracy")
    
    best_f1_idx = df_summary['F1-Score'].idxmax()
    best_f1 = df_summary.loc[best_f1_idx]
    print(f"🎯 Best F1-Score: {best_f1['Model']} with {best_f1['F1-Score']:.4f} F1")
    
    df_summary['Accuracy per Million Params'] = df_summary['Accuracy'] / (df_summary['Total Parameters'] / 1e6)
    best_efficient_idx = df_summary['Accuracy per Million Params'].idxmax()
    best_efficient = df_summary.loc[best_efficient_idx]
    print(f"⚡ Most Parameter-Efficient: {best_efficient['Model']} "
          f"({best_efficient['Accuracy per Million Params']:.6f} accuracy per million parameters)")
    
    fastest_idx = df_summary['Avg Inference Time/sample (ms)'].idxmin()
    fastest = df_summary.loc[fastest_idx]
    print(f"🚀 Fastest Inference: {fastest['Model']} with {fastest['Avg Inference Time/sample (ms)']:.2f} ms per sample")
    
    df_summary['Accuracy per MS'] = df_summary['Accuracy'] / df_summary['Avg Inference Time/sample (ms)']
    best_tradeoff_idx = df_summary['Accuracy per MS'].idxmax()
    best_tradeoff = df_summary.loc[best_tradeoff_idx]
    print(f"⚖️  Best Accuracy-Speed Trade-off: {best_tradeoff['Model']} "
          f"({best_tradeoff['Accuracy per MS']:.4f} accuracy per ms)")
    
    print("="*120)
    
    detailed_path = os.path.join(Config.results_dir, "detailed_comparison.json")
    with open(detailed_path, 'w') as f:
        json.dump({k: {key: val for key, val in v.items() 
                      if key not in ['predictions', 'labels', 'class_report', 'train_losses', 'train_accuracies']} 
                  for k, v in successful_models.items()}, 
                 f, indent=2, default=str)
    
    return df_summary

if __name__ == "__main__":
    print("="*100)
    print("LIGHTWEIGHT CODE CLASSIFICATION MODEL COMPARISON FRAMEWORK")
    print("="*100)
    print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Device: {Config.device}")
    print(f"Number of Models to Compare: {len(Config.models_to_test)}")
    print(f"Number of Classes: {num_classes}")
    print("="*100)
    
    all_results = run_all_experiments()
    create_comparison_plots(all_results)
    df_summary = generate_summary_report(all_results)
    
    print(f"\nExperiment completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Results saved to: {Config.results_dir}")
    print(f"Models saved to: {Config.models_dir}")